# Naturral Scenes Classification

### Imports

In [47]:
import random
from typing import Tuple, Dict
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

### Set Seed

In [48]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

### Data Loading

In [49]:
def get_xray_loaders(
    dataDir: str,
    seed: int,
    batch_size: int = 32,
) -> Tuple[DataLoader, DataLoader]:
    set_seed(seed)
 
    transform = T.Compose([
        T.Resize((64, 64)),
        T.ToTensor()
    ])
 
    fullDataset = torchvision.datasets.ImageFolder(root=dataDir, transform=transform)
 
    trainIndices = []
    testIndices  = []
 
    for classIdx in range(len(fullDataset.classes)):
 
        classIndices = []
        for i in range(len(fullDataset.samples)):
            _, label = fullDataset.samples[i]
            if label == classIdx:
                classIndices.append(i)
 
        random.shuffle(classIndices)
 
        splitIdx = int(len(classIndices) * 0.7)
 
        trainIndices += classIndices[:splitIdx]
        testIndices  += classIndices[splitIdx:]
 
    trainDataset = Subset(fullDataset, trainIndices)
    testDataset  = Subset(fullDataset, testIndices)
 
    train_loader = DataLoader(trainDataset, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(testDataset,  batch_size=batch_size, shuffle=False)
 
    print(f"Train: {len(trainDataset)} | Test: {len(testDataset)}")
    return train_loader, test_loader

In [50]:
train_loader, test_loader = get_xray_loaders(dataDir="data/", seed=0)

Train: 3659 | Test: 1569


### MLP Model

In [51]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes=(512, 256), dropout_p=0.3):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1  = nn.Linear(12288, hidden_sizes[0])
        self.layer2  = nn.Linear(hidden_sizes[0], hidden_sizes[1])
        self.layer3  = nn.Linear(hidden_sizes[1], 3)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_p)
 
    def forward(self, x):
        x = self.flatten(x)
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer3(x)
        return x

### Training

In [52]:
def train_one_epoch(model, loader, optimizer, device) -> Dict[str, float]:
    model.train()
    lossFunction = nn.CrossEntropyLoss(label_smoothing=0.1)
    totalLoss = 0
    correct   = 0
    samples   = 0
 
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
 
        outputs = model(images)
        loss = lossFunction(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        totalLoss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        samples += images.size(0)
 
    return {"loss": totalLoss / samples, "acc": correct / samples}

In [53]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
 
set_seed(0)
model     = MLP(hidden_sizes=(512, 256), dropout_p=0.3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
 
EPOCHS = 20
for epoch in range(1, EPOCHS + 1):
    trainMetrics = train_one_epoch(model, train_loader, optimizer, device)
    print(f"Epoch {epoch}/{EPOCHS} | Loss: {trainMetrics['loss']:.4f} | Acc: {trainMetrics['acc']:.4f}")

Using device: cpu
Epoch 1/20 | Loss: 0.8368 | Acc: 0.6704
Epoch 2/20 | Loss: 0.5957 | Acc: 0.8355
Epoch 3/20 | Loss: 0.5898 | Acc: 0.8439
Epoch 4/20 | Loss: 0.5559 | Acc: 0.8595
Epoch 5/20 | Loss: 0.5484 | Acc: 0.8653
Epoch 6/20 | Loss: 0.5336 | Acc: 0.8748
Epoch 7/20 | Loss: 0.5316 | Acc: 0.8715
Epoch 8/20 | Loss: 0.5232 | Acc: 0.8773
Epoch 9/20 | Loss: 0.5548 | Acc: 0.8491
Epoch 10/20 | Loss: 0.5495 | Acc: 0.8593
Epoch 11/20 | Loss: 0.5311 | Acc: 0.8669
Epoch 12/20 | Loss: 0.5373 | Acc: 0.8705
Epoch 13/20 | Loss: 0.5150 | Acc: 0.8770
Epoch 14/20 | Loss: 0.5267 | Acc: 0.8628
Epoch 15/20 | Loss: 0.5169 | Acc: 0.8737
Epoch 16/20 | Loss: 0.5330 | Acc: 0.8609
Epoch 17/20 | Loss: 0.5180 | Acc: 0.8789
Epoch 18/20 | Loss: 0.5169 | Acc: 0.8732
Epoch 19/20 | Loss: 0.5202 | Acc: 0.8726
Epoch 20/20 | Loss: 0.5236 | Acc: 0.8653


### Evaluation

In [54]:
@torch.no_grad()
def evaluate(model, loader, device) -> Dict[str, float]:
    model.eval()
    lossFunction = nn.CrossEntropyLoss(label_smoothing=0.1)
    totalLoss = 0
    correct   = 0
    samples   = 0
 
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
 
        outputs = model(images)
        loss    = lossFunction(outputs, labels)
 
        totalLoss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        samples += images.size(0)
 
    return {"loss": totalLoss / samples, "acc": correct / samples}

In [55]:
testMetrics = evaluate(model, test_loader, device)
print(f"Test Loss: {testMetrics['loss']:.4f} | Test Acc: {testMetrics['acc']:.4f}")

Test Loss: 0.4740 | Test Acc: 0.9178
